# E蛋白主动学习工作流：4折交叉验证集成 + MC Dropout并行推理 (v4)

--- 

**目标**: 利用4折交叉验证训练出的模型检查点，结合蒙特卡洛Dropout，以并行计算的方式高效地对饱和突变库进行不确定性评估。

**工作流程**: 
1.  **生成突变数据**: 首先，执行**步骤一**的单元格，为E蛋白的WT、T9I和T11A背景生成饱和突变数据池（`.pkl`文件）。如果这些文件已存在，可以跳过此步。
2.  **并行推理**: 执行**步骤二**的单元格。此单元格是核心部分，它会：
    * 自动查找`k1`到`k4`文件夹中`validation_pearson`分数最高的`.ckpt`文件。
    * 创建4个并行进程，将每个模型的推理任务分配到一个独立的GPU上（例如，k1 -> cuda:0, k2 -> cuda:1 ...）。
    * 在每个进程中，模型会进行25次MC Dropout前向传播，并将结果**保存到临时文件**。
3.  **结果分析与保存**: 主进程会等待所有子进程完成，然后**读取临时文件**来收集所有结果（4个模型 * 25次 = 100个预测/突变体），计算均值和方差，并最终将详细报告保存到指定的Excel文件中。

---

In [ ]:
import pandas as pd
df = pd.read_csv("/home/zhoukaitao/mnt/hdd1/virus_dataset/flu/extract/flu_segment_meta.csv")


/tmp/ipykernel_1691467/500094258.py:2: DtypeWarning: Columns (18,34) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("/home/zhoukaitao/mnt/hdd1/virus_dataset/flu/extract/flu_segment_meta.csv")


,id,isolate,isolate_name,data_source,ispublic,segment_accession,reference,reference_detailed,species,genus,...,collection_date_year,collection_date_month,collection_date_day,submission_year,submission_month,submission_day,length,clade,update_date,year_month
0,2529,A/American Blue-winged teal/TX/22-029888-002-o...,A/American Blue-winged teal/TX/22-029888-002-o...,NCBI,True,PQ701302,A-seg1,CY002079.1,Alphainfluenzavirus influenzae,Alphainfluenzavirus,...,2022,9,11.0,2024,12,6,2280,NaN,2025-02-21,2022-09
1,2530,A/American Blue-winged teal/TX/22-029888-002-o...,A/American Blue-winged teal/TX/22-029888-002-o...,NCBI,True,PQ701303,A-seg2,CY003646.1,Alphainfluenzavirus influenzae,Alphainfluenzavirus,...,2022,9,11.0,2024,12,6,2274,NaN,2025-02-21,2022-09
2,2531,A/American Blue-winged teal/TX/22-029888-002-o...,A/American Blue-winged teal/TX/22-029888-002-o...,NCBI,True,PQ701304,A-seg3,CY003645.1,Alphainfluenzavirus influenzae,Alphainfluenzavirus,...,2022,9,11.0,2024,12,6,2151,NaN,2025-02-21,2022-09
3,2532,A/American Blue-winged teal/TX/22-029888-002-o...,A/American Blue-winged teal/TX/22-029888-002-o...,NCBI,True,PQ701305,A-seg4_H5,DQ864721.1,Alphainfluenzavirus influenzae,Alphainfluenzavirus,...,2022,9,11.0,2024,12,6,1704,2.3.4.4b,2025-02-21,2022-09
4,2533,A/American Blue-winged teal/TX/22-029888-002-o...,A/American Blue-winged teal/TX/22-029888-002-o...,NCBI,True,PQ701306,A-seg5,CY006079.1,Alphainfluenzavirus influenzae,Alphainfluenzavirus,...,2022,9,11.0,2024,12,6,1497,NaN,2025-02-21,2022-09
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
319963,1340699,GCA_050106575.1,A/Sao Paulo/SPPU08647/2021(H3N2)_1,NCBI,True,PP271659,A-seg4_H3,CY002000.1,Alphainfluenzavirus influenzae,Alphainfluenzavirus,...,2021,12,13.0,2025,5,1,1762,3C.2a1b.2a.2a.3,NaN,2021-12
319964,1340700,GCA_050106575.1,A/Sao Paulo/SPPU08647/2021(H3N2)_1,NCBI,True,PP271660,A-seg5,CY006079.1,Alphainfluenzavirus influenzae,Alphainfluenzavirus,...,2021,12,13.0,2025,5,1,1566,NaN,NaN,2021-12
319965,1340701,GCA_050106575.1,A/Sao Paulo/SPPU08647/2021(H3N2)_1,NCBI,True,PP271661,A-seg6_N2,CY002010.1,Alphainfluenzavirus influenzae,Alphainfluenzavirus,...,2021,12,13.0,2025,5,1,1467,B.3,NaN,2021-12
319966,1340702,GCA_050106575.1,A/Sao Paulo/SPPU08647/2021(H3N2)_1,NCBI,True,PP271662,A-seg7,CY002009.1,Alphainfluenzavirus influenzae,Alphainfluenzavirus,...,2021,12,13.0,2025,5,1,1027,NaN,NaN,2021-12


In [11]:
df.head(50)

,id,isolate,isolate_name,data_source,ispublic,segment_accession,reference,reference_detailed,species,genus,...,collection_date_year,collection_date_month,collection_date_day,submission_year,submission_month,submission_day,length,clade,update_date,year_month
0,2529,A/American Blue-winged teal/TX/22-029888-002-o...,A/American Blue-winged teal/TX/22-029888-002-o...,NCBI,True,PQ701302,A-seg1,CY002079.1,Alphainfluenzavirus influenzae,Alphainfluenzavirus,...,2022,9,11.0,2024,12,6,2280,NaN,2025-02-21,2022-09
1,2530,A/American Blue-winged teal/TX/22-029888-002-o...,A/American Blue-winged teal/TX/22-029888-002-o...,NCBI,True,PQ701303,A-seg2,CY003646.1,Alphainfluenzavirus influenzae,Alphainfluenzavirus,...,2022,9,11.0,2024,12,6,2274,NaN,2025-02-21,2022-09
2,2531,A/American Blue-winged teal/TX/22-029888-002-o...,A/American Blue-winged teal/TX/22-029888-002-o...,NCBI,True,PQ701304,A-seg3,CY003645.1,Alphainfluenzavirus influenzae,Alphainfluenzavirus,...,2022,9,11.0,2024,12,6,2151,NaN,2025-02-21,2022-09
3,2532,A/American Blue-winged teal/TX/22-029888-002-o...,A/American Blue-winged teal/TX/22-029888-002-o...,NCBI,True,PQ701305,A-seg4_H5,DQ864721.1,Alphainfluenzavirus influenzae,Alphainfluenzavirus,...,2022,9,11.0,2024,12,6,1704,2.3.4.4b,2025-02-21,2022-09
4,2533,A/American Blue-winged teal/TX/22-029888-002-o...,A/American Blue-winged teal/TX/22-029888-002-o...,NCBI,True,PQ701306,A-seg5,CY006079.1,Alphainfluenzavirus influenzae,Alphainfluenzavirus,...,2022,9,11.0,2024,12,6,1497,NaN,2025-02-21,2022-09
5,2534,A/American Blue-winged teal/TX/22-029888-002-o...,A/American Blue-winged teal/TX/22-029888-002-o...,NCBI,True,PQ701307,A-seg6_N1,CY002538.1,Alphainfluenzavirus influenzae,Alphainfluenzavirus,...,2022,9,11.0,2024,12,6,1410,NaN,2025-02-21,2022-09
6,2535,A/American Blue-winged teal/TX/22-029888-002-o...,A/American Blue-winged teal/TX/22-029888-002-o...,NCBI,True,PQ701308,A-seg7,CY002009.1,Alphainfluenzavirus influenzae,Alphainfluenzavirus,...,2022,9,11.0,2024,12,6,982,NaN,2025-02-21,2022-09
7,2536,A/American Blue-winged teal/TX/22-029888-002-o...,A/American Blue-winged teal/TX/22-029888-002-o...,NCBI,True,PQ701309,A-seg8,CY002284.1,Alphainfluenzavirus influenzae,Alphainfluenzavirus,...,2022,9,11.0,2024,12,6,838,NaN,2025-02-21,2022-09
8,6922,A/Arizona/MAP6087A/2017(H3N2)_1,A/Arizona/MAP6087A/2017(H3N2)_1,NCBI,True,MH700972,A-seg4_H3,CY002000.1,Alphainfluenzavirus influenzae,Alphainfluenzavirus,...,2017,12,13.0,2018,8,5,1701,3C.2a2,2025-02-21,2017-12
9,6923,A/Arizona/MAP6087A/2017(H3N2)_1,A/Arizona/MAP6087A/2017(H3N2)_1,NCBI,True,MH700973,A-seg7,CY002009.1,Alphainfluenzavirus influenzae,Alphainfluenzavirus,...,2017,12,13.0,2018,8,5,982,NaN,2025-02-21,2017-12


## 步骤一：生成候选突变体数据池

**说明**: 此单元格用于生成评估所需的`.pkl`文件。如果`/data2/zhoukaitao/01evoModel/dataset/251015_E_Saturat/`目录下已存在`E_WT_saturation_mutants.pkl`等文件，您可以**直接跳到步骤二**。

In [ ]:
# 导入必要的库
import pickle
import os
import json
import torch
from tqdm.notebook import tqdm

# 导入您的项目模块
import ioutils
import trainUtils
from esm.sdk.api import ESMProtein

# --- 💡 用户配置区 --- #

# 1. 输入文件路径
FASTA_PATH = "fasta/E.fasta"
PDB_PATH = "pdb/E.pdb"

# 2. 用于编码PDB文件的模型配置文件路径 (使用k1的即可)
ENCODER_CONFIG_PATH = "/data2/zhoukaitao/01evoModel/checkpoints/251015_E_Drop/k1/config.json"

# 3. 输出目录
OUTPUT_DIR = "/data2/zhoukaitao/01evoModel/dataset/251015_E_Saturat"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# --- 辅助函数 --- #
AMINO_ACIDS = "ACDEFGHIKLMNPQRSTVWY"

def generate_saturation_mutagenesis(base_sequence: str, background_name: str):
    """对给定的基础序列进行饱和突变扫描"""
    mutants = []
    seq_list = list(base_sequence)
    for i in range(len(base_sequence)):
        original_aa = seq_list[i]
        for new_aa in AMINO_ACIDS:
            if original_aa == new_aa:
                continue
            
            mutant_seq_list = seq_list[:]
            mutant_seq_list[i] = new_aa
            mutant_sequence = "".join(mutant_seq_list)
            mutation_id = f"{background_name}_{original_aa}{i+1}{new_aa}"
            
            mutants.append({
                "id": mutation_id,
                "sequence": mutant_sequence
            })
    return mutants

# --- 主逻辑 --- #
print("开始生成突变体数据...")

try:
    with open(ENCODER_CONFIG_PATH, 'r') as f:
        encoder_configs = json.load(f)
    esm_encoder_model = trainUtils.loadPretrainModel(encoder_configs)
    esm_encoder_model.eval()
    print("ESM编码器模型加载成功。")

    fasta_gen = ioutils.readFasta(FASTA_PATH)
    _, WT_SEQUENCE = next(fasta_gen)
    print(f"从 '{FASTA_PATH}' 加载WT序列成功，长度: {len(WT_SEQUENCE)}")

    ref_protein_obj = ESMProtein.from_pdb(PDB_PATH)
    with torch.no_grad():
        encoded_ref = esm_encoder_model.encode(ref_protein_obj)
        ref_seq_tokens = encoded_ref.sequence.tolist()
        ref_structure_tokens = encoded_ref.structure.tolist()
    print(f"从 '{PDB_PATH}' 加载并编码参考序列和结构成功。")

    MUTATION_BACKGROUNDS = {
        "E_WT": WT_SEQUENCE,
        "E_T9I": WT_SEQUENCE[:8] + "I" + WT_SEQUENCE[9:],
        "E_T11A": WT_SEQUENCE[:10] + "A" + WT_SEQUENCE[11:],
    }

    for name, sequence in MUTATION_BACKGROUNDS.items():
        output_path = os.path.join(OUTPUT_DIR, f"{name}_saturation_mutants.pkl")
        if os.path.exists(output_path):
            print(f"\n文件 '{output_path}' 已存在，跳过生成。")
            continue

        print(f"\n--- 正在处理背景: {name} ---")
        mutants_info = generate_saturation_mutagenesis(sequence, name)
        print(f"为 {name} 背景生成了 {len(mutants_info)} 个单点突变体。")
        
        unlabeled_pool = []
        for mutant in tqdm(mutants_info, desc=f"Tokenizing {name}"):
            mutant_protein_obj = ESMProtein(sequence=mutant['sequence'])
            with torch.no_grad():
                encoded_mutant = esm_encoder_model.encode(mutant_protein_obj)
                mutant_seq_tokens = encoded_mutant.sequence.tolist()
            
            sample = {
                "id": mutant["id"],
                "E": {"seq_t": mutant_seq_tokens, "structure_t": ref_structure_tokens},
                "aligned_E": {"seq_t": ref_seq_tokens, "structure_t": ref_structure_tokens}
            }
            unlabeled_pool.append(sample)
            
        with open(output_path, "wb") as f:
            pickle.dump(unlabeled_pool, f, protocol=pickle.HIGHEST_PROTOCOL)
        print(f"已将 {len(unlabeled_pool)} 个突变体数据保存到: {output_path}")

    print("\n所有突变池生成完毕！")

except Exception as e:
    print(f"\n发生错误: {e}")
    print("请检查所有文件路径是否正确，并且相关文件存在。")


## 步骤二：并行推理与不确定性评估

**说明**: 此单元格是核心。它将启动多个进程，在不同的GPU上并行执行推理任务。请确保您的环境中安装了`torch`, `pandas`, `tqdm`, 和 `openpyxl`。

In [ ]:
import torch
import numpy as np
import pandas as pd
import json
import pickle
import glob
import os
import re
import shutil
from tqdm.notebook import tqdm
from torch.utils.data import DataLoader
import torch.multiprocessing as mp

# 导入您的项目模块
import trainUtils
from VirusDataset import VESMDataset

# --- 💡 用户配置区 --- #
BASE_CHECKPOINT_DIR = "/data2/zhoukaitao/01evoModel/checkpoints/251015_E_Drop/"
FOLDS = ["k1", "k2", "k3", "k4"] # 交叉验证文件夹
MUTANT_POOL_GLOB_PATTERN = "/data2/zhoukaitao/01evoModel/dataset/251015_E_Saturat/E_*_saturation_mutants.pkl"
OUTPUT_EXCEL_PATH = "/data2/zhoukaitao/01evoModel/dataset/251015_E_Saturat/ensemble_mutant_uncertainty_results.xlsx"

# --- 推理参数 --- #
N_PASSES_PER_MODEL = 25
BATCH_SIZE = 400
DEVICES = [f"cuda:{i}" for i in [0,2,3,4]] # 为每个模型分配一个GPU
PREDICTION_NAMES = ["Expression", "CCK8", "Activation"]

# --- 辅助函数 --- #
def find_best_ckpt(fold_dir):
    """在指定文件夹中查找validation_pearson分数最高的ckpt文件""" 
    ckpt_files = glob.glob(os.path.join(fold_dir, "*.ckpt"))
    if not ckpt_files:
        raise FileNotFoundError(f"在 '{fold_dir}' 中找不到任何 .ckpt 文件。")
    
    best_ckpt = None
    max_pearson = -float('inf')
    
    for ckpt in ckpt_files:
        match = re.search(r"validation_pearson=([\d\.]+)\.ckpt", os.path.basename(ckpt))
        if match:
            pearson_score = float(match.group(1))
            if pearson_score > max_pearson:
                max_pearson = pearson_score
                best_ckpt = ckpt
                
    if best_ckpt is None:
        raise ValueError(f"在 '{fold_dir}' 的文件名中找不到 'validation_pearson' 分数。")
        
    return best_ckpt, max_pearson

def inference_worker(rank, fold, config_path, best_ckpt_path, all_unlabeled_data, temp_dir):
    """并行推理的工作进程函数，将结果保存到文件"""
    device = DEVICES[rank]
    from tqdm import tqdm
    print(f"进程 {rank} ({fold}): 开始在 {device} 上进行推理...")
    
    try:
        with open(config_path, "r") as f:
            configs = json.load(f)
        
        pretrain_model = trainUtils.loadPretrainModel(configs)
        model = trainUtils.buildModel(configs, pretrain_model, best_ckpt_path)
        model.to(device)
        model.eval()
        if configs["model"]["params"].get("regressor_version") == "mc_dropout":
            model.enable_mc_dropout()

        dataset = VESMDataset([all_unlabeled_data], "inference", configs["dataset"]["seq"], train_time_series=False)
        loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

        fold_predictions = []
        with torch.no_grad():
            for _ in tqdm(range(N_PASSES_PER_MODEL), desc=f'{fold} on {device}', position=rank, leave=False):
                pass_predictions = []
                for batch in loader:
                    input_data = batch["input"]
                    for k, v in input_data.items():
                        if isinstance(v, dict):
                            for vk, vv in v.items():
                                v[vk] = torch.tensor(vv, dtype=torch.long).to(device) if isinstance(vv, list) else vv.to(device)
                    
                    output = model.forward(input_data)
                    preds = output.S1Predicts['E']['predictions']
                    pass_predictions.append(preds.cpu())
                fold_predictions.append(torch.cat(pass_predictions, dim=0))
        
        all_ids = [sample['id'] for sample in all_unlabeled_data]
        output_path = os.path.join(temp_dir, f"{fold}_results.pt")
        torch.save({'preds': torch.stack(fold_predictions), 'ids': all_ids}, output_path)
        print(f"进程 {rank} ({fold}): 推理完成，结果已保存。")
    except Exception as e:
        import traceback
        print(f"进程 {rank} ({fold}) 发生错误: {e}\n{traceback.format_exc()}")

# --- 主编排逻辑 --- #
if __name__ == '__main__':
    try:
        mp.set_start_method('spawn', force=True)
        print("Multiprocessing start method set to 'spawn'.")
    except RuntimeError:
        pass
        
    print("--- 步骤A: 加载所有突变体数据 ---")
    all_mutant_files = glob.glob(MUTANT_POOL_GLOB_PATTERN)
    if not all_mutant_files:
        print(f"错误: 在'{MUTANT_POOL_GLOB_PATTERN}'下找不到任何突变体数据文件。请先运行步骤一。")
    else:
        all_unlabeled_data = []
        for file_path in sorted(all_mutant_files):
            print(f"  - 加载 {os.path.basename(file_path)}")
            with open(file_path, "rb") as f:
                all_unlabeled_data.extend(pickle.load(f))
        print(f"总共加载了 {len(all_unlabeled_data)} 个突变体进行评估。")
        
        print("\n--- 步骤B: 准备并行推理任务 ---")
        processes = []
        temp_output_dir = "./temp_inference_results"
        if os.path.exists(temp_output_dir): shutil.rmtree(temp_output_dir)
        os.makedirs(temp_output_dir)
        
        for rank, fold in enumerate(FOLDS):
            fold_dir = os.path.join(BASE_CHECKPOINT_DIR, fold)
            config_path = os.path.join(fold_dir, 'config.json')
            best_ckpt, score = find_best_ckpt(fold_dir)
            print(f"  - {fold}: 找到最佳ckpt '{os.path.basename(best_ckpt)}' (pearson={score:.4f})")
            
            p = mp.Process(target=inference_worker, args=(rank, fold, config_path, best_ckpt, all_unlabeled_data, temp_output_dir))
            processes.append(p)
            
        print("\n--- 步骤C: 启动并行推理 (这可能需要较长时间) ---")
        for p in processes:
            p.start()
        
        # 等待所有进程完成
        for p in tqdm(processes, desc="等待模型完成"):
            p.join()
        
        print("\n--- 步骤D: 合并结果并计算统计数据 ---")
        result_files = glob.glob(os.path.join(temp_output_dir, "*_results.pt"))
        if len(result_files) != len(FOLDS):
            print(f"\n错误: 预期有 {len(FOLDS)} 个结果文件，但只找到 {len(result_files)} 个。请检查上面的错误信息。")
        else:
            all_preds_list = []
            final_ids = None
            for res_file in sorted(result_files):
                data = torch.load(res_file)
                all_preds_list.append(data['preds'])
                if final_ids is None: final_ids = data['ids']
            
            final_preds_tensor = torch.cat(all_preds_list, dim=0)
            mean_preds = final_preds_tensor.mean(dim=0).numpy()
            variance_preds = final_preds_tensor.var(dim=0).numpy()

            results_df = pd.DataFrame({
                'mutation_id': final_ids,
                'background': [mid.split('_')[1] for mid in final_ids]
            })
            
            for i, name in enumerate(PREDICTION_NAMES):
                results_df[f'{name}_pred_mean'] = mean_preds[:, i]
                results_df[f'{name}_uncertainty_var'] = variance_preds[:, i]
            
            results_df['total_uncertainty_var'] = variance_preds.sum(axis=1)
            results_df = results_df.sort_values(by='total_uncertainty_var', ascending=False).reset_index(drop=True)
            
            results_df.to_excel(OUTPUT_EXCEL_PATH, index=False, engine='openpyxl')
            print(f"成功！最终结果已保存到 '{OUTPUT_EXCEL_PATH}'")
            
            print("\n结果预览:")
            display(results_df.head())
            
            # 清理临时文件 (已注释掉)
            # shutil.rmtree(temp_output_dir)
            print(f"临时推理结果已保留在文件夹: '{temp_output_dir}'")

In [1]:
import os

def get_amino_acids_at_positions(fasta_path, positions):
    """
    读取FASTA文件并查找特定位置的氨基酸。
    """
    if not os.path.exists(fasta_path):
        print(f"错误: 找不到文件 {fasta_path}")
        return

    # 1. 读取FASTA序列
    sequence = []
    with open(fasta_path, 'r') as f:
        header = None
        for line in f:
            line = line.strip()
            if not line:
                continue
            if line.startswith('>'):
                # 如果已经读取了第一条序列的数据且遇到了第二个header，则停止
                if header is not None and sequence:
                    break
                header = line
            else:
                sequence.append(line)
    
    full_sequence = "".join(sequence)
    seq_len = len(full_sequence)
    print(f"序列读取完成。Header: {header}")
    print(f"序列总长度: {seq_len}")
    print("-" * 30)

    # 2. 查找指定位置
    print(f"{'Position (1-based)':<20} {'Amino Acid'}")
    print("-" * 30)
    
    for pos in positions:
        index = pos - 1  # 将1-based位置转换为0-based索引
        
        if 0 <= index < seq_len:
            aa = full_sequence[index]
            print(f"{pos:<20} {aa}")
        else:
            print(f"{pos:<20} [越界! 最大位置为 {seq_len}]")

# --- 配置 ---
fasta_file_path = 'fasta/S.fasta'
target_positions = [493, 22, 59, 190, 445]

# --- 执行 ---
get_amino_acids_at_positions(fasta_file_path, target_positions)

序列读取完成。Header: >YP_009724390.1 surface glycoprotein [Severe acute respiratory syndrome coronavirus 2]
序列总长度: 1273
------------------------------
Position (1-based)   Amino Acid
------------------------------
493                  Q
22                   T
59                   F
190                  R
445                  V
